# CS 3110/3990: Data Privacy
## In-Class Exercise #4, week of 9/28/2026

In [2]:
# Load the data and libraries
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

def laplace_mech(v, sensitivity, epsilon):
    return v + np.random.laplace(loc=0, scale=sensitivity / epsilon)

def pct_error(orig, priv):
    return np.abs(orig - priv)/orig * 100.0

adult = pd.read_csv('https://github.com/jnear/cs3110-data-privacy/raw/main/homework/adult_with_pii.csv')

## Question 1

For various values of $b$, write code to print out the percent error of summing the ages in the `adult` dataset, 
clipped to each value of $b$.

In [3]:
bs = range(1, 100, 10)
real_sum = adult['Age'].sum()

for b in bs:
    clipped_sum = adult['Age'].clip(lower=0, upper = b).sum()
    print(f"% error for {b} is {pct_error(real_sum, clipped_sum)}")

% error for 1 is 97.4080940444511
% error for 11 is 71.48903448896206
% error for 21 is 46.000380495392264
% error for 31 is 25.509031989473492
% error for 41 is 11.7959939725709
% error for 51 is 4.329687317165198
% error for 61 is 1.1587597123836921
% error for 71 is 0.2341877497996031
% error for 81 is 0.03860675005194001
% error for 91 is 0.0


What value of $b$ is the best?

Any value of b > than 81 is fine, at b = 81 the relative error due to bias is small. Values much larger than this risk adding attitional error when we add noise to achieve DP. 

## Question 2

For various values of $b$, print the result of a *differentially private* sum of ages, clipped to each value of $b$. Use $\epsilon = 0.1$.

In [4]:
bs = range(1, 100, 10)
real_sum = adult['Age'].sum()

# If pct_error is removed, DP is satisfied
# For a total privacy cost of ep = 10 * 0.1 by sequential compostion
for b in bs:
    clipped_sum = adult['Age'].clip(lower=0, upper = b).sum()
    dp_sum=laplace_mech(clipped_sum, sensitivity=b, epsilon=0.1)
    print(f"fDP result for {b} is {dp_sum}, error is {pct_error(real_sum, dp_sum)}")

fDP result for 1 is 32538.568754780597, error is 97.40987960626045
fDP result for 11 is 358488.969489649, error is 71.46372362584654
fDP result for 21 is 678925.8637395903, error is 45.95645128826425
fDP result for 31 is 935546.5723891708, error is 25.52904601612801
fDP result for 41 is 1108012.4720026415, error is 11.800493688581117
fDP result for 51 is 1201865.2455272875, error is 4.329667772813407
fDP result for 61 is 1241982.8834645592, error is 1.1362417511258276
fDP result for 71 is 1253675.7463286575, error is 0.20547178414468595
fDP result for 81 is 1258925.6469298478, error is 0.21242842267527864
fDP result for 91 is 1256994.1992042393, error is 0.05868219673516612


Which value of $b$ is the best now? Does it differ?

It does not really differ, 81 or 91 is still good, the error due to bias is much greater than the error due to variance, until the clipping param is really big

## Question 3

Write an algorithm to *automatically pick the clipping parameter* for the age column. Your algorithm should satisfy differential privacy.


Look for a bunch of canidates for b, like Q2
Find thte first pair(b, b') st noisy val for b > noisy val for b'

In [5]:
def pick_b(epsilon):
    bs = range(1, 200, 20)
    real_sum = adult['Age'].sum()
    epsilon_i = epsilon / len(bs)
    dp_sums = []
    last_sum = 0
    for b in bs: #Satisfies len(bs) * epsilon_i-DP by sequential comp
        clipped_sum = adult['Age'].clip(lower=0, upper = b).sum()
        dp_sum = laplace_mech(clipped_sum, sensitivity=b, epsilon=epsilon_i)
        if dp_sum < last_sum:
            return b
        else:
            last_sum = dp_sum
        # dp_sum=laplace_mech(clipped_sum, sensitivity=b, epsilon=0.1)
        # print(f"fDP result for {b} is {dp_sum}")
    
    # raise Exception("No good val!")
    return bs[-1]

pick_b(1.0)
result = laplace_mech(adult['Age'].clip(lower=0, upper= b), sensitivity=b, epsilon=2.0)

In [6]:
# TEST CASE for question 3

many_trials = np.mean([pick_b(1.0) for _ in range(100)])
assert many_trials > 70
assert many_trials < 500

In [ ]:
# s, so the result is DP by post-processing
noisy_sum = laplace_mech(adult['Age'].sum(), sensitivity=120, epsilon=1.99) #assuming people < 120
# result = noisy_sum / len(adult) # BAD, len needs noise
noisy_count = laplace_mech(adult['Age'], sensitivity=1, epsilon=0.01)
result = noisy_sum / noisy_count


## Question 4

What is the privacy cost of your algorithm, and why?

In [ ]:
# 1 I use laplace mech twice. For sum, the s is 120 because I assume nobody is > than 120 and the sum can increase by at most 120
# Count s is 1 because it's a counting query
# 2 Composition: I use laplace mech twice. The eps are 1.99 and 0.01, so the total privacy cost sums to 2 by seq comp.
# 3 Post Processing: My return value is noisy_sum / noisy_count. Both inputs are result of DP mech

The privacy cost of pick_b is 'len(bs) * epsilon_i = epsilon'

Each iter fof the loop uses ep i privacy cost and there are len(bs) iters

## Question 5

Write code to generate a *histogram* of education numbers in the `adult` dataset.

In [13]:
def education_hist():
    hist =  adult['Education'].value_counts()
    return hist

education_hist()

Education
HS-grad         10501
Some-college     7291
Bachelors        5355
Masters          1723
Assoc-voc        1382
11th             1175
Assoc-acdm       1067
10th              933
7th-8th           646
Prof-school       576
9th               514
12th              433
Doctorate         413
5th-6th           333
1st-4th           168
Preschool          51
Name: count, dtype: int64

In [14]:
# TEST CASE for question 5
h = education_hist()
assert h['HS-grad'] == 10501
assert h['12th'] == 433
assert h['Doctorate'] == 413

## Question 6

Write code to release a *differentially private histogram* of education numbers.

In [15]:
def dp_education_hist(epsilon):
    hist =  adult['Education'].value_counts()
    epsilon_i = epsilon / len(hist)
    noisy_hist = hist.apply(lambda x: laplace_mech(x, sensitivity=1, epsilon=epsilon))
    return noisy_hist

dp_education_hist(1.0)

Education
HS-grad         10501.475486
Some-college     7290.325201
Bachelors        5354.059854
Masters          1721.597026
Assoc-voc        1380.652245
11th             1174.938408
Assoc-acdm       1064.542166
10th              933.095912
7th-8th           643.731448
Prof-school       576.446398
9th               512.966097
12th              433.650569
Doctorate         412.447720
5th-6th           334.375944
1st-4th           169.743684
Preschool          50.116201
Name: count, dtype: float64

In [16]:
# TEST CASE for question 6
h = dp_education_hist(1.0)
assert abs(h['HS-grad'] - 10501) < 100
assert abs(h['Doctorate'] - 413) < 100

## Question 7

What is the total privacy cost of `dp_education_hist`, and why?

1: Sensitivity: Sensitivity is 1 because everything is a counting query
2: Comp: Total priv cost is len(hist) * epsilon_i = epsilon
3: Post processing: All elements have noise added

## Question 8

Write code to generate a differentially private *contingency table* for the Education and Sex columns of the `adult` dataset.

In [ ]:
def dp_crosstab_education_sex(epsilon):
    # YOUR CODE HERE
    raise NotImplementedError()

dp_crosstab_education_sex(1.0)

In [ ]:
# TEST CASE for question 8
ct = dp_crosstab_education_sex(1.0)
assert abs(ct['Female']['10th'] - 295) < 100
assert abs(ct['Male']['10th'] - 638) < 100
assert abs(ct['Female']['Bachelors'] - 1619) < 100

assert abs(ct['Female']['10th'] - 295) > 0
assert abs(ct['Male']['10th'] - 638) > 0
assert abs(ct['Female']['Bachelors'] - 1619) > 0

## Question 9

Does parallel composition apply for the contingency table in question 1? Why or why not?

YOUR ANSWER HERE

## Question 10

Does the number of columns used in constructing the contingency table matter for privacy cost? Does it matter for accuracy?

YOUR ANSWER HERE